# Advanced Nested Context Managers and `ExitStack`
## Problems with Complete Solutions

This notebook develops production-quality patterns for nested and dynamically acquired resources.

Covered topics:

- nested enter/exit ordering
- dynamic file management
- partial-acquisition rollback
- `ExitStack.enter_context`
- `ExitStack.callback`
- `ExitStack.push`
- `ExitStack.pop_all`
- conditional context managers
- exception propagation, suppression, and transformation
- reusable composite managers
- `AsyncExitStack`
- a simplified exit-stack implementation

Every main problem includes a specification, a complete solution, executable tests, and design notes.

> Target runtime: Python 3.10+

## Learning objectives

After completing this notebook, you should be able to:

1. Predict exact entry and cleanup order.
2. Manage a runtime-defined number of resources safely.
3. Clean up resources when acquisition fails halfway through.
4. Register cleanup for APIs that are not context managers.
5. Transfer cleanup ownership with `pop_all()`.
6. Compose optional synchronous and asynchronous resources.
7. Preserve correct exception semantics while unwinding.
8. Recognize when a handwritten stack is unsafe.

## Best-practice checklist

- Use a regular `with` statement when the set of managers is fixed and small.
- Use `ExitStack` when resources are dynamic, conditional, or acquired in a loop.
- Acquire managers with `stack.enter_context(...)`.
- Register ordinary cleanup functions with `stack.callback(...)`.
- Register `__exit__`-style callables with `stack.push(...)`.
- Expect cleanup in last-in, first-out order.
- Test normal exit, body failure, and acquisition failure.
- Avoid manually calling `__enter__` and `__exit__` in application code.
- Do not ignore boolean suppression results from nested `__exit__` methods.
- Use `AsyncExitStack` when asynchronous cleanup is required.

## Shared imports and helpers

In [1]:
from __future__ import annotations

import asyncio
import io
import os
import tempfile
from contextlib import (
    AsyncExitStack,
    ExitStack,
    asynccontextmanager,
    contextmanager,
    nullcontext,
)
from dataclasses import dataclass, field
from itertools import zip_longest
from pathlib import Path
from typing import Any, AsyncIterator, Callable, Iterable, Iterator, Sequence, TextIO

In [2]:
@contextmanager
def traced_resource(
    name: str,
    events: list[str],
    *,
    fail_on_enter: bool = False,
    suppress: tuple[type[BaseException], ...] = (),
) -> Iterator[str]:
    events.append(f"enter:{name}")

    if fail_on_enter:
        events.append(f"enter-failed:{name}")
        raise RuntimeError(f"could not acquire {name}")

    try:
        yield name
    except BaseException as exc:
        events.append(f"exit-exception:{name}:{type(exc).__name__}")
        if isinstance(exc, suppress):
            events.append(f"suppressed:{name}")
            return
        raise
    else:
        events.append(f"exit-normal:{name}")
    finally:
        events.append(f"cleanup:{name}")

# Problem 1 — Prove nested enter/exit order

Create three traced managers and verify:

- entry occurs left to right
- exit occurs right to left
- cleanup occurs after a body exception
- the same exception reaches each manager unless one suppresses or replaces it

In [3]:
events: list[str] = []

try:
    with traced_resource("A", events) as a:
        with traced_resource("B", events) as b:
            with traced_resource("C", events) as c:
                assert (a, b, c) == ("A", "B", "C")
                events.append("body")
                raise ValueError("body failed")
except ValueError:
    events.append("caught")

expected = [
    "enter:A",
    "enter:B",
    "enter:C",
    "body",
    "exit-exception:C:ValueError",
    "cleanup:C",
    "exit-exception:B:ValueError",
    "cleanup:B",
    "exit-exception:A:ValueError",
    "cleanup:A",
    "caught",
]

assert events == expected
events

['enter:A',
 'enter:B',
 'enter:C',
 'body',
 'exit-exception:C:ValueError',
 'cleanup:C',
 'exit-exception:B:ValueError',
 'cleanup:B',
 'exit-exception:A:ValueError',
 'cleanup:A',
 'caught']

### Key idea

Nested managers form a stack. The last successfully entered manager is the first one asked to exit.

# Problem 2 — Replace indentation pyramids

Demonstrate both:

1. one `with` statement containing several fixed managers
2. `ExitStack` for a runtime-generated collection

In [4]:
events = []

with (
    traced_resource("A", events) as a,
    traced_resource("B", events) as b,
    traced_resource("C", events) as c,
):
    events.append(f"body:{a},{b},{c}")

assert events == [
    "enter:A",
    "enter:B",
    "enter:C",
    "body:A,B,C",
    "exit-normal:C",
    "cleanup:C",
    "exit-normal:B",
    "cleanup:B",
    "exit-normal:A",
    "cleanup:A",
]

events

['enter:A',
 'enter:B',
 'enter:C',
 'body:A,B,C',
 'exit-normal:C',
 'cleanup:C',
 'exit-normal:B',
 'cleanup:B',
 'exit-normal:A',
 'cleanup:A']

In [5]:
events = []

with ExitStack() as stack:
    values = [
        stack.enter_context(traced_resource(name, events))
        for name in ("A", "B", "C")
    ]
    events.append("body:" + ",".join(values))

assert values == ["A", "B", "C"]
assert events[-6:] == [
    "exit-normal:C",
    "cleanup:C",
    "exit-normal:B",
    "cleanup:B",
    "exit-normal:A",
    "cleanup:A",
]

events

['enter:A',
 'enter:B',
 'enter:C',
 'body:A,B,C',
 'exit-normal:C',
 'cleanup:C',
 'exit-normal:B',
 'cleanup:B',
 'exit-normal:A',
 'cleanup:A']

### Selection rule

Use fixed syntax when the resources are known while writing the code. Use `ExitStack` when the collection is created at runtime.

# Problem 3 — Zip a dynamic number of text files

Implement `zip_text_files(paths, delimiter=",", mode="shortest")`.

Modes:

- `shortest`: stop when the first file ends
- `longest`: continue to the longest file and fill missing fields with `""`
- `strict`: raise `ValueError` when line counts differ

All files must close on every exit path.

In [6]:
def zip_text_files(
    paths: Sequence[str | os.PathLike[str]],
    *,
    delimiter: str = ",",
    mode: str = "shortest",
    encoding: str = "utf-8",
) -> list[str]:
    if not paths:
        return []

    if mode not in {"shortest", "longest", "strict"}:
        raise ValueError("mode must be 'shortest', 'longest', or 'strict'")

    with ExitStack() as stack:
        streams = [
            stack.enter_context(open(path, "r", encoding=encoding))
            for path in paths
        ]

        if mode == "shortest":
            rows = zip(*streams)
        elif mode == "longest":
            rows = zip_longest(*streams, fillvalue="")
        else:
            rows = zip(*streams, strict=True)

        return [
            delimiter.join(field.rstrip("\r\n") for field in row)
            for row in rows
        ]

In [7]:
with tempfile.TemporaryDirectory() as temp_dir:
    root = Path(temp_dir)
    p1 = root / "a.txt"
    p2 = root / "b.txt"
    p3 = root / "c.txt"

    p1.write_text("a1\na2\na3\n", encoding="utf-8")
    p2.write_text("b1\nb2\n", encoding="utf-8")
    p3.write_text("c1\nc2\nc3\n", encoding="utf-8")

    shortest = zip_text_files([p1, p2, p3], mode="shortest")
    longest = zip_text_files([p1, p2, p3], mode="longest")

    assert shortest == ["a1,b1,c1", "a2,b2,c2"]
    assert longest == ["a1,b1,c1", "a2,b2,c2", "a3,,c3"]

    try:
        zip_text_files([p1, p2, p3], mode="strict")
    except ValueError:
        strict_failed = True
    else:
        strict_failed = False

    assert strict_failed

shortest, longest

(['a1,b1,c1', 'a2,b2,c2'], ['a1,b1,c1', 'a2,b2,c2', 'a3,,c3'])

### Best-practice detail

Use `rstrip("\r\n")` when removing line endings. Plain `strip()` may remove meaningful spaces or tabs.

# Problem 4 — Partial acquisition failure

Force the third acquisition to fail and verify:

- earlier resources are cleaned up
- later resources are never entered
- cleanup remains LIFO

In [8]:
events = []

try:
    with ExitStack() as stack:
        for name in ("A", "B", "C", "D"):
            stack.enter_context(
                traced_resource(
                    name,
                    events,
                    fail_on_enter=(name == "C"),
                )
            )
except RuntimeError as exc:
    assert "could not acquire C" in str(exc)

assert events == [
    "enter:A",
    "enter:B",
    "enter:C",
    "enter-failed:C",
    "exit-exception:B:RuntimeError",
    "cleanup:B",
    "exit-exception:A:RuntimeError",
    "cleanup:A",
]

events

['enter:A',
 'enter:B',
 'enter:C',
 'enter-failed:C',
 'exit-exception:B:RuntimeError',
 'cleanup:B',
 'exit-exception:A:RuntimeError',
 'cleanup:A']

### Why this matters

Each successful `enter_context` immediately registers its cleanup. A later failure therefore unwinds everything already acquired.

# Problem 5 — Optional ownership with `nullcontext`

Implement `write_report(destination=None)`.

- Path: open and own the file.
- Existing stream: use but do not close it.
- `None`: create an in-memory buffer.
- Return the generated text.

In [9]:
def write_report(
    destination: str | os.PathLike[str] | TextIO | None = None,
) -> str:
    if destination is None:
        buffer = io.StringIO()
        manager = nullcontext(buffer)
        read_back = lambda: buffer.getvalue()
    elif isinstance(destination, (str, os.PathLike)):
        path = Path(destination)
        manager = open(path, "w", encoding="utf-8")
        read_back = lambda: path.read_text(encoding="utf-8")
    else:
        manager = nullcontext(destination)
        read_back = lambda: destination.getvalue()

    with manager as stream:
        stream.write("name,value\n")
        stream.write("alpha,10\n")
        stream.write("beta,20\n")

    return read_back()

In [10]:
memory_text = write_report()
assert "alpha,10" in memory_text

external = io.StringIO()
external_text = write_report(external)
assert external.closed is False
assert external_text == external.getvalue()

with tempfile.TemporaryDirectory() as temp_dir:
    output = Path(temp_dir) / "report.csv"
    path_text = write_report(output)
    assert path_text == output.read_text(encoding="utf-8")

memory_text

'name,value\nalpha,10\nbeta,20\n'

# Problem 6 — Register cleanup for a non-context-manager API

Create a fake service that exposes `start()` and `stop()` but no context-manager protocol. Use `ExitStack.callback` to guarantee shutdown.

In [11]:
@dataclass
class Service:
    name: str
    events: list[str]
    running: bool = False

    def start(self) -> None:
        if self.running:
            raise RuntimeError(f"{self.name} already running")
        self.running = True
        self.events.append(f"start:{self.name}")

    def stop(self) -> None:
        if self.running:
            self.running = False
            self.events.append(f"stop:{self.name}")

In [12]:
events = []
services = [Service(name, events) for name in ("cache", "queue", "worker")]

try:
    with ExitStack() as stack:
        for service in services:
            service.start()
            stack.callback(service.stop)
        events.append("body")
        raise LookupError("force cleanup")
except LookupError:
    pass

assert events == [
    "start:cache",
    "start:queue",
    "start:worker",
    "body",
    "stop:worker",
    "stop:queue",
    "stop:cache",
]
assert all(not service.running for service in services)

events

['start:cache',
 'start:queue',
 'start:worker',
 'body',
 'stop:worker',
 'stop:queue',
 'stop:cache']

### `callback` versus `push`

`callback` registers an ordinary callable and cannot suppress exceptions. `push` registers a callable with the `__exit__(exc_type, exc, tb)` signature, so it can inspect or suppress an exception.

# Problem 7 — Register an `__exit__`-style cleanup with `push`

Create a cleanup function that suppresses only `KeyError` and records what it observed.

In [13]:
events = []


def suppress_key_error(exc_type, exc, traceback) -> bool:
    observed = exc_type.__name__ if exc_type is not None else "None"
    events.append(f"push-observed:{observed}")
    return exc_type is not None and issubclass(exc_type, KeyError)


with ExitStack() as stack:
    stack.push(suppress_key_error)
    events.append("body")
    raise KeyError("optional item missing")

assert events == ["body", "push-observed:KeyError"]
events

['body', 'push-observed:KeyError']

# Problem 8 — Selective suppression inside a stack

Place three managers in an `ExitStack`. The innermost manager suppresses `ValueError`.

Verify that outer managers receive a normal exit after suppression.

In [14]:
events = []

with ExitStack() as stack:
    stack.enter_context(traced_resource("outer", events))
    stack.enter_context(traced_resource("middle", events))
    stack.enter_context(
        traced_resource("inner", events, suppress=(ValueError,))
    )
    events.append("body")
    raise ValueError("handled by inner")

assert events == [
    "enter:outer",
    "enter:middle",
    "enter:inner",
    "body",
    "exit-exception:inner:ValueError",
    "suppressed:inner",
    "cleanup:inner",
    "exit-normal:middle",
    "cleanup:middle",
    "exit-normal:outer",
    "cleanup:outer",
]

events

['enter:outer',
 'enter:middle',
 'enter:inner',
 'body',
 'exit-exception:inner:ValueError',
 'suppressed:inner',
 'cleanup:inner',
 'exit-normal:middle',
 'cleanup:middle',
 'exit-normal:outer',
 'cleanup:outer']

### Exception-state mutation

When an inner exit callback suppresses an exception, later outer callbacks are invoked as if the body exited normally. A correct stack must update the active exception state while unwinding.

# Problem 9 — Transform an exception during unwinding

Create a manager that converts `OSError` into `ResourceBundleError`. Place an outer tracer around it and prove the outer tracer sees the replacement exception.

In [15]:
class ResourceBundleError(RuntimeError):
    pass


@contextmanager
def translate_os_error(label: str) -> Iterator[None]:
    try:
        yield
    except OSError as exc:
        raise ResourceBundleError(f"{label}: {exc}") from exc

In [16]:
events = []

try:
    with ExitStack() as stack:
        stack.enter_context(traced_resource("outer", events))
        stack.enter_context(translate_os_error("dataset load"))
        raise FileNotFoundError("records.csv")
except ResourceBundleError as exc:
    assert isinstance(exc.__cause__, FileNotFoundError)
    events.append("caught:ResourceBundleError")

assert "exit-exception:outer:ResourceBundleError" in events
events

['enter:outer',
 'exit-exception:outer:ResourceBundleError',
 'cleanup:outer',
 'caught:ResourceBundleError']

# Problem 10 — Transfer cleanup ownership with `pop_all()`

Acquire resources in a temporary stack, validate them, then transfer cleanup responsibility to a returned stack.

This pattern is useful when construction and lifetime ownership happen in different layers.

In [17]:
@dataclass
class ResourceBundle:
    values: list[str]
    cleanup_stack: ExitStack

    def close(self) -> None:
        self.cleanup_stack.close()


def acquire_bundle(names: Sequence[str], events: list[str]) -> ResourceBundle:
    with ExitStack() as stack:
        values = [
            stack.enter_context(traced_resource(name, events))
            for name in names
        ]

        if len(set(values)) != len(values):
            raise ValueError("resource names must be unique")

        transferred = stack.pop_all()
        return ResourceBundle(values=values, cleanup_stack=transferred)

In [18]:
events = []
bundle = acquire_bundle(["db", "cache", "queue"], events)

assert bundle.values == ["db", "cache", "queue"]
assert events == ["enter:db", "enter:cache", "enter:queue"]

bundle.close()

assert events[-6:] == [
    "exit-normal:queue",
    "cleanup:queue",
    "exit-normal:cache",
    "cleanup:cache",
    "exit-normal:db",
    "cleanup:db",
]

events

['enter:db',
 'enter:cache',
 'enter:queue',
 'exit-normal:queue',
 'cleanup:queue',
 'exit-normal:cache',
 'cleanup:cache',
 'exit-normal:db',
 'cleanup:db']

### Ownership rule

After `pop_all()`, the original stack no longer owns those callbacks. The returned stack must eventually be closed.

# Problem 11 — Transactional setup with rollback callbacks

Build a configuration installer that creates several files. If any validation step fails, remove every file created so far. On success, keep them.

Use `stack.callback` for rollback and `pop_all()` to cancel rollback after commit.

In [19]:
def install_configuration(
    directory: Path,
    files: dict[str, str],
    *,
    fail_after: int | None = None,
) -> list[Path]:
    directory.mkdir(parents=True, exist_ok=True)
    created: list[Path] = []

    with ExitStack() as rollback:
        for index, (name, content) in enumerate(files.items(), start=1):
            path = directory / name
            path.write_text(content, encoding="utf-8")
            created.append(path)
            rollback.callback(path.unlink, missing_ok=True)

            if fail_after == index:
                raise ValueError(f"validation failed after {name}")

        rollback.pop_all()

    return created

In [20]:
with tempfile.TemporaryDirectory() as temp_dir:
    root = Path(temp_dir)

    try:
        install_configuration(
            root,
            {"a.ini": "A", "b.ini": "B", "c.ini": "C"},
            fail_after=2,
        )
    except ValueError:
        pass

    assert list(root.iterdir()) == []

    created = install_configuration(
        root,
        {"a.ini": "A", "b.ini": "B", "c.ini": "C"},
    )

    assert [path.name for path in created] == ["a.ini", "b.ini", "c.ini"]
    assert all(path.exists() for path in created)

print("Rollback and commit behavior verified.")

Rollback and commit behavior verified.


# Problem 12 — Build a reusable composite context manager

Implement `open_text_group(paths)` as a class-based manager backed internally by `ExitStack`.

Requirements:

- support a dynamic list of paths
- expose entered streams
- close everything on normal or exceptional exit
- reject re-entry of the same instance

In [21]:
class OpenTextGroup:
    def __init__(
        self,
        paths: Iterable[str | os.PathLike[str]],
        *,
        encoding: str = "utf-8",
    ) -> None:
        self.paths = [Path(path) for path in paths]
        self.encoding = encoding
        self._stack: ExitStack | None = None
        self.streams: list[TextIO] = []

    def __enter__(self) -> list[TextIO]:
        if self._stack is not None:
            raise RuntimeError("this OpenTextGroup instance is already active")

        stack = ExitStack()
        self._stack = stack

        try:
            self.streams = [
                stack.enter_context(open(path, "r", encoding=self.encoding))
                for path in self.paths
            ]
        except BaseException:
            stack.close()
            self._stack = None
            self.streams = []
            raise

        return self.streams

    def __exit__(self, exc_type, exc, traceback) -> bool:
        assert self._stack is not None
        stack = self._stack
        self._stack = None
        try:
            return stack.__exit__(exc_type, exc, traceback)
        finally:
            self.streams = []

In [22]:
with tempfile.TemporaryDirectory() as temp_dir:
    root = Path(temp_dir)
    paths = []

    for index in range(3):
        path = root / f"part-{index}.txt"
        path.write_text(f"value-{index}", encoding="utf-8")
        paths.append(path)

    group = OpenTextGroup(paths)

    with group as streams:
        assert [stream.read() for stream in streams] == [
            "value-0", "value-1", "value-2"
        ]
        assert all(not stream.closed for stream in streams)

    assert all(stream.closed for stream in streams)

print("Composite manager verified.")

Composite manager verified.


# Problem 13 — Implement a simplified exit stack

Build `MiniExitStack` for learning purposes.

It must:

- enter context managers
- register callbacks
- unwind in reverse order
- pass the active exception to each exit callback
- update exception state after suppression

Use the standard-library `ExitStack` in production.

In [23]:
class MiniExitStack:
    def __init__(self) -> None:
        self._callbacks: list[Callable[[Any, Any, Any], bool]] = []

    def __enter__(self) -> "MiniExitStack":
        return self

    def enter_context(self, manager: Any) -> Any:
        value = manager.__enter__()
        self._callbacks.append(manager.__exit__)
        return value

    def callback(
        self,
        function: Callable[..., Any],
        *args: Any,
        **kwargs: Any,
    ) -> Callable[..., Any]:
        def exit_callback(exc_type, exc, traceback) -> bool:
            function(*args, **kwargs)
            return False

        self._callbacks.append(exit_callback)
        return function

    def push(
        self,
        exit_callback: Callable[[Any, Any, Any], bool],
    ) -> Callable[[Any, Any, Any], bool]:
        self._callbacks.append(exit_callback)
        return exit_callback

    def __exit__(self, exc_type, exc, traceback) -> bool:
        received_exception = exc_type is not None
        suppressed = False

        while self._callbacks:
            callback = self._callbacks.pop()
            if callback(exc_type, exc, traceback):
                suppressed = True
                exc_type = exc = traceback = None

        return received_exception and suppressed

In [24]:
events = []

with MiniExitStack() as stack:
    stack.enter_context(traced_resource("A", events))
    stack.enter_context(
        traced_resource("B", events, suppress=(KeyError,))
    )
    stack.callback(events.append, "plain-callback")
    events.append("body")
    raise KeyError("suppressed")

assert events == [
    "enter:A",
    "enter:B",
    "body",
    "plain-callback",
    "exit-exception:B:KeyError",
    "suppressed:B",
    "cleanup:B",
    "exit-normal:A",
    "cleanup:A",
]

events

['enter:A',
 'enter:B',
 'body',
 'plain-callback',
 'exit-exception:B:KeyError',
 'suppressed:B',
 'cleanup:B',
 'exit-normal:A',
 'cleanup:A']

### Limitations of the educational implementation

The real `ExitStack` handles exception replacement, traceback details, callback decorators, ownership transfer, and subtle exception-chain behavior. Reimplementing it is useful for understanding the protocol, not for replacing it.

# Problem 14 — Asynchronous dynamic resources with `AsyncExitStack`

Create several asynchronous connections at runtime and register an ordinary asynchronous cleanup callback.

In [25]:
@dataclass
class AsyncConnection:
    name: str
    events: list[str]
    closed: bool = False

    async def query(self, statement: str) -> str:
        if self.closed:
            raise RuntimeError("connection is closed")
        await asyncio.sleep(0)
        return f"{self.name}:{statement}"


@asynccontextmanager
async def open_async_connection(
    name: str,
    events: list[str],
) -> AsyncIterator[AsyncConnection]:
    events.append(f"async-enter:{name}")
    connection = AsyncConnection(name=name, events=events)

    try:
        yield connection
    finally:
        await asyncio.sleep(0)
        connection.closed = True
        events.append(f"async-exit:{name}")

In [26]:
async def async_stack_demo() -> list[str]:
    events: list[str] = []

    async def final_audit(message: str) -> None:
        await asyncio.sleep(0)
        events.append(message)

    async with AsyncExitStack() as stack:
        connections = [
            await stack.enter_async_context(
                open_async_connection(name, events)
            )
            for name in ("primary", "replica", "analytics")
        ]

        stack.push_async_callback(final_audit, "async-callback")

        results = [
            await connection.query("SELECT 1")
            for connection in connections
        ]
        events.append("results:" + ",".join(results))

    assert all(connection.closed for connection in connections)
    return events


async_events = await async_stack_demo()
assert async_events[-4:] == [
    "async-callback",
    "async-exit:analytics",
    "async-exit:replica",
    "async-exit:primary",
]

async_events

['async-enter:primary',
 'async-enter:replica',
 'async-enter:analytics',
 'results:primary:SELECT 1,replica:SELECT 1,analytics:SELECT 1',
 'async-callback',
 'async-exit:analytics',
 'async-exit:replica',
 'async-exit:primary']

# Additional solved drills

## Drill A — Conditional debugging resource

Use `nullcontext` so the same code path works whether debugging is enabled or disabled.

In [27]:
@contextmanager
def debug_trace(events: list[str]) -> Iterator[None]:
    events.append("debug-enter")
    try:
        yield
    finally:
        events.append("debug-exit")


def run_operation(debug: bool) -> list[str]:
    events: list[str] = []
    manager = debug_trace(events) if debug else nullcontext()

    with manager:
        events.append("operation")

    return events


assert run_operation(False) == ["operation"]
assert run_operation(True) == ["debug-enter", "operation", "debug-exit"]

## Drill B — Close an arbitrary collection of objects

Use `stack.callback` with objects that expose only `close()`.

In [28]:
@dataclass
class Closable:
    name: str
    events: list[str]
    closed: bool = False

    def close(self) -> None:
        if not self.closed:
            self.closed = True
            self.events.append(f"close:{self.name}")


events = []
objects = [Closable(str(index), events) for index in range(4)]

with ExitStack() as stack:
    for obj in objects:
        stack.callback(obj.close)
    events.append("body")

assert events == ["body", "close:3", "close:2", "close:1", "close:0"]
assert all(obj.closed for obj in objects)

## Drill C — Keep selected resources and close the rest

Acquire several resources, transfer all cleanup, then manually retain ownership of the chosen bundle.

In [29]:
events = []

with ExitStack() as temporary:
    values = [
        temporary.enter_context(traced_resource(name, events))
        for name in ("one", "two", "three")
    ]
    retained_stack = temporary.pop_all()

assert values == ["one", "two", "three"]
assert events == ["enter:one", "enter:two", "enter:three"]

retained_stack.close()
assert events[-6:] == [
    "exit-normal:three",
    "cleanup:three",
    "exit-normal:two",
    "cleanup:two",
    "exit-normal:one",
    "cleanup:one",
]

# Failure-mode laboratory

The following anti-pattern loses exception-suppression semantics because it calls every nested `__exit__` method with the original exception and always returns `False`.

In [30]:
class BrokenNestedContexts:
    def __init__(self, *managers: Any) -> None:
        self.managers = managers
        self.exits: list[Callable[..., bool]] = []
        self.values: list[Any] = []

    def __enter__(self) -> list[Any]:
        for manager in self.managers:
            self.values.append(manager.__enter__())
            self.exits.append(manager.__exit__)
        return self.values

    def __exit__(self, exc_type, exc, traceback) -> bool:
        for exit_method in reversed(self.exits):
            exit_method(exc_type, exc, traceback)
        return False

In [31]:
events = []

try:
    with BrokenNestedContexts(
        traced_resource("outer", events),
        traced_resource("inner", events, suppress=(KeyError,)),
    ):
        raise KeyError("should have been suppressed")
except KeyError:
    leaked = True
else:
    leaked = False

assert leaked is True
assert "suppressed:inner" in events
print("The inner manager attempted suppression, but the broken wrapper leaked the exception.")

The inner manager attempted suppression, but the broken wrapper leaked the exception.


### Lesson

A correct composite manager must preserve the full exit protocol. It is not enough to call cleanup methods in reverse order.

# Final challenge — A dynamic processing pipeline

Compose:

- a variable number of input files
- an optional audit stream
- rollback for an output file
- cleanup transfer after successful completion

The function writes rows formed by zipping all inputs. If processing fails, the partial output is removed.

In [32]:
def build_combined_file(
    inputs: Sequence[Path],
    output: Path,
    *,
    audit: TextIO | None = None,
) -> int:
    output.parent.mkdir(parents=True, exist_ok=True)

    with ExitStack() as stack:
        streams = [
            stack.enter_context(open(path, "r", encoding="utf-8"))
            for path in inputs
        ]
        destination = stack.enter_context(
            open(output, "w", encoding="utf-8", newline="")
        )

        rollback = stack.enter_context(ExitStack())
        rollback.callback(output.unlink, missing_ok=True)

        audit_manager = nullcontext(audit) if audit is not None else nullcontext()
        audit_stream = stack.enter_context(audit_manager)

        count = 0
        for row in zip(*streams, strict=True):
            cleaned = [field.rstrip("\r\n") for field in row]
            destination.write(",".join(cleaned) + "\n")
            count += 1

            if audit_stream is not None:
                audit_stream.write(f"wrote-row:{count}\n")

        destination.flush()
        os.fsync(destination.fileno())

        rollback.pop_all()
        return count

In [33]:
with tempfile.TemporaryDirectory() as temp_dir:
    root = Path(temp_dir)
    inputs = []

    for index, prefix in enumerate(("a", "b", "c"), start=1):
        path = root / f"input-{index}.txt"
        path.write_text(f"{prefix}1\n{prefix}2\n", encoding="utf-8")
        inputs.append(path)

    output = root / "combined.csv"
    audit = io.StringIO()

    count = build_combined_file(inputs, output, audit=audit)

    assert count == 2
    assert output.read_text(encoding="utf-8") == "a1,b1,c1\na2,b2,c2\n"
    assert audit.getvalue() == "wrote-row:1\nwrote-row:2\n"

print("Final pipeline completed successfully.")

Final pipeline completed successfully.


# Review questions with answers

### 1. Why does cleanup run in reverse order?

Later resources often depend on earlier resources. Reversing the acquisition order removes dependents before dependencies.

### 2. What is the difference between `enter_context`, `callback`, and `push`?

- `enter_context` enters a context manager and registers its `__exit__`.
- `callback` registers an ordinary callable that cannot suppress exceptions.
- `push` registers an `__exit__`-style callable that can inspect or suppress exceptions.

### 3. Why is partial acquisition a major reason to use `ExitStack`?

Each successful acquisition is registered immediately, so a later failure unwinds everything already acquired.

### 4. What does `pop_all()` do?

It transfers all registered callbacks to a new stack without executing them.

### 5. When should `nullcontext` be used?

When one branch needs a real manager and another branch should do nothing while preserving one shared `with` structure.

### 6. Why is a simple reverse loop over `__exit__` methods insufficient?

Because each exit callback can suppress or replace the active exception, changing what outer callbacks must receive.

### 7. When is `AsyncExitStack` required?

When acquisition or cleanup includes asynchronous context managers or asynchronous callbacks.

# Summary

Reliable dynamic resource management follows a small set of rules:

1. Register cleanup immediately after each successful acquisition.
2. Unwind in last-in, first-out order.
3. Preserve exception suppression and transformation semantics.
4. Use `callback` for plain cleanup and `push` for exit-style cleanup.
5. Transfer ownership explicitly with `pop_all()`.
6. Prefer standard-library stacks over handwritten infrastructure.
7. Test success, body failure, and acquisition failure.